# AIML Lab 7 - Medical Reviews Analysis from Social Media Data

Dataset: `medical_reviews.csv`, a balanced sample of 3600 patient reviews from Drugs.com.
The 1 to 10 star rating was mapped to a sentiment label (>= 8 Positive, <= 4 Negative, 5 to 7 Neutral).

## Part A - Import and Explore Medical Review Data

In [ ]:
import re
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("medical_reviews.csv")
print("Dataset Imported Successfully.")
df.head()

In [ ]:
print("Number of Medical Reviews:", df.shape[0])
print("Number of Features:", df.shape[1])
print("Columns:", list(df.columns))

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
counts = df["Sentiment"].value_counts()

print("Sentiment Class Distribution:")
print("Positive :", counts["Positive"])
print("Negative :", counts["Negative"])
print("Neutral  :", counts["Neutral"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x="Sentiment", data=df, order=["Negative", "Neutral", "Positive"],
              hue="Sentiment", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Sentiment class distribution")

sns.histplot(df["Rating"], bins=10, ax=axes[1])
axes[1].set_title("Star rating distribution")

plt.tight_layout()
plt.show()

In [ ]:
print("Most reviewed conditions:")
print(df["Condition"].value_counts().head(10).to_string())

In [ ]:
reviews = df["Review"]
labels = df["Sentiment"]
print(reviews.shape, labels.shape)

## Part B - Text Preprocessing

In [ ]:
before = len(df)
df = df.dropna(subset=["Review"])
df = df[df["Review"].str.strip() != ""]
removed = before - len(df)

print("TEXT PREPROCESSING")
print("-" * 40)
print("Missing Reviews Removed:", removed)
print("Reviews remaining:", len(df))

In [ ]:
stop_words = set(stopwords.words("english"))
# negations are kept, dropping them would flip the meaning of a review
stop_words -= {"not", "no", "nor", "never", "against", "don't", "doesn't", "didn't", "won't"}


def clean_text(text):
    text = html.unescape(text)             # the reviews contain &#039; and &amp;
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    return " ".join(t for t in tokens if t not in stop_words and len(t) > 1)

In [ ]:
demo = "The doctor was AMAZING!!! Visit https://hospital.com @patient #GoodHospital"
print("Original:", demo)
print("Cleaned :", clean_text(demo))

In [ ]:
df["Cleaned_Review"] = df["Review"].apply(clean_text)
df = df[df["Cleaned_Review"] != ""]

for i in [0, 5, 12]:
    print("Original Review:")
    print(df["Review"].iloc[i][:220])
    print()
    print("Cleaned Review:")
    print(df["Cleaned_Review"].iloc[i][:220])
    print("-" * 60)

print("Text Preprocessing Completed Successfully.")

In [ ]:
df["word_count"] = df["Cleaned_Review"].str.split().str.len()
print(df["word_count"].describe().round(1).to_string())
df[["Review", "Cleaned_Review", "Sentiment"]].head()

## Part C - TF-IDF Feature Extraction

In [ ]:
X = df["Cleaned_Review"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)     # fitted on the training data only
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF FEATURE EXTRACTION")
print("-" * 40)
print("Training Reviews:", X_train.shape[0])
print("Testing Reviews :", X_test.shape[0])
print()
print("Number of TF-IDF Features:", len(tfidf.get_feature_names_out()))
print()
print("Training Matrix Shape:", X_train_tfidf.shape)
print("Testing Matrix Shape :", X_test_tfidf.shape)
print()
print("TF-IDF Feature Extraction Completed.")

In [ ]:
feature_names = tfidf.get_feature_names_out()
print("Sample unigrams:", [f for f in feature_names if " " not in f][:15])
print()
print("Sample bigrams :", [f for f in feature_names if " " in f][:15])

## Part D - Train and Compare Classification Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}

print("MODEL COMPARISON")
print("-" * 40)
for name, model in models.items():
    scores = cross_validate(model, X_train_tfidf, y_train, cv=cv,
                            scoring=["accuracy", "f1_macro"])
    cv_scores[name] = scores
    print()
    print(name)
    print(f"Mean CV Accuracy : {scores['test_accuracy'].mean():.4f}")
    print(f"Mean CV F1-Score : {scores['test_f1_macro'].mean():.4f}")

best_name = max(cv_scores, key=lambda n: cv_scores[n]["test_f1_macro"].mean())
print()
print("Selected Model:", best_name)

In [ ]:
for model in models.values():
    model.fit(X_train_tfidf, y_train)

best_model = models[best_name]

## Part E - Predict Sentiment of a New Medical Review

In [ ]:
new_review = ("The doctor explained the treatment clearly and the hospital staff "
              "were very supportive.")

cleaned = clean_text(new_review)
vector = tfidf.transform([cleaned])
prediction = best_model.predict(vector)[0]
probs = best_model.predict_proba(vector)[0]

print("NEW MEDICAL REVIEW CLASSIFICATION")
print("-" * 40)
print("Review:")
print(new_review)
print()
print("Cleaned:", cleaned)
print()
print("Predicted Sentiment:", prediction)
print()
print("Class Probabilities:")
for cls, p in zip(best_model.classes_, probs):
    print(f"{cls:<9}: {p:.4f}")

## Part F - Evaluate the Selected Model

In [ ]:
y_pred = best_model.predict(X_test_tfidf)

print("FINAL MODEL EVALUATION")
print("-" * 40)
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='macro'):.4f}")

In [ ]:
print("CLASSIFICATION REPORT")
print("-" * 40)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=best_model.classes_)
print("Confusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
recalls = recall_score(y_test, y_pred, average=None, labels=best_model.classes_)
recall_by_class = pd.Series(recalls, index=best_model.classes_).round(4)

print(recall_by_class.to_string())
print()
print("Highest recall:", recall_by_class.idxmax(), f"({recall_by_class.max():.4f})")
print("Lowest recall :", recall_by_class.idxmin(), f"({recall_by_class.min():.4f})")

In [ ]:
comparison = []
for name, model in models.items():
    p = model.predict(X_test_tfidf)
    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, p),
        "Precision": precision_score(y_test, p, average="macro"),
        "Recall": recall_score(y_test, p, average="macro"),
        "F1-Score": f1_score(y_test, p, average="macro"),
    })

pd.DataFrame(comparison).set_index("Model").round(4)

## Part G - Analyse Important Words

In [ ]:
lr = models["Logistic Regression"]

print("IMPORTANT FEATURES")
for idx, sentiment in enumerate(lr.classes_):
    top = np.argsort(lr.coef_[idx])[-10:][::-1]
    print()
    print(f"{sentiment} Sentiment:")
    for rank, j in enumerate(top, 1):
        print(f"{rank:>2}. {feature_names[j]:<22} {lr.coef_[idx][j]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, idx in zip(axes, range(len(lr.classes_))):
    top = np.argsort(lr.coef_[idx])[-10:]
    ax.barh([feature_names[j] for j in top], lr.coef_[idx][top])
    ax.set_title(lr.classes_[idx])

plt.tight_layout()
plt.show()

## Part H - Industry-Oriented Application

Patient Reviews / Social Media Posts -> Data Collection -> Text Cleaning -> Text Preprocessing
-> TF-IDF Features -> Trained ML Classifier -> Sentiment Classification -> Healthcare Analytics
-> Human Review

In [ ]:
def analyse_feedback(reviews):
    vectors = tfidf.transform([clean_text(r) for r in reviews])
    preds = best_model.predict(vectors)
    conf = best_model.predict_proba(vectors).max(axis=1)
    return pd.DataFrame({
        "Review": [r[:70] + "..." if len(r) > 70 else r for r in reviews],
        "Sentiment": preds,
        "Confidence": conf.round(3),
        "Action": ["escalate to service quality team" if p == "Negative"
                   else "no action needed" for p in preds],
    })


incoming = [
    "This medication changed my life, the side effects were minimal and my symptoms improved within a week.",
    "Terrible experience, I had severe nausea and dizziness and had to stop taking it after three days.",
    "I have been on this for a month. Some improvement but also some headaches, so hard to say yet.",
    "Waited four hours in the OPD and nobody bothered to update us @cityhospital",
    "The doctor was thorough and the nursing staff took great care of me throughout my stay.",
]

feedback = analyse_feedback(incoming)
feedback

In [ ]:
summary = feedback["Sentiment"].value_counts()
print("Batch of", len(feedback), "reviews")
print(summary.to_string())
print()
print("Negative feedback rate:", f"{summary.get('Negative', 0) / len(feedback) * 100:.1f} %")